<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 2: Working with Text Data

Packages that are being used in this notebook:
<br><small>แพ็กเกจ (ไลบรารี) ที่ถูกนำมาใช้งานในสมุดบันทึกโค้ด (Notebook) นี้:

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.1
tiktoken version: 0.7.0


- This chapter covers data preparation and sampling to get input data "ready" for the LLM
- <small>บทนี้ครอบคลุมเรื่องการเตรียมข้อมูลและการสุ่มตัวอย่างข้อมูล เพื่อจัดเตรียมข้อมูลนำเข้าให้ "พร้อม" สำหรับการใช้งานกับ LLM 

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="1450px">

&nbsp;
## 2.1 Understanding word embeddings

- There are many forms of embeddings; we focus on text embeddings in this book
- การฝังข้อมูล (Embeddings) มีหลากหลายรูปแบบ แต่ในหนังสือเล่มนี้เราจะมุ่งเน้นไปที่การฝังข้อมูลประเภทข้อความ (Text embeddings) เป็นหลัก

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="1450px">

- LLMs work with embeddings in high-dimensional spaces (i.e., thousands of dimensions)
- Since we can't visualize such high-dimensional spaces (we humans think in 1, 2, or 3 dimensions), the figure below illustrates a 2-dimensional embedding space
---
- โมเดล LLM ทำงานกับข้อมูล Embeddings ในพื้นที่มิติสูง (เช่น หลายพันมิติ)
- เนื่องจากเราไม่สามารถจินตนาการภาพของพื้นที่มิติสูงขนาดนั้นได้ (มนุษย์เราคิดและมองเห็นภาพได้แค่ในรูปแบบ 1, 2 หรือ 3 มิติเท่านั้น) รูปภาพด้านล่างนี้จึงแสดงตัวอย่างจำลองของพื้นที่ Embedding ในรูปแบบ 2 มิติแทน

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="800px">

&nbsp;
## 2.2 Tokenizing text
*(การแบ่งข้อความออกเป็นโทเค็น หรือ Tokenization) คือกระบวนการทางคอมพิวเตอร์ที่นำข้อความดิบ (Raw Text) เช่น ประโยคหรือบทความ มาตัดแบ่งออกเป็นหน่วยย่อย ๆ ที่เรียกว่า โทเค็น (Token)*

- In this section, we tokenize text, which means breaking text into smaller units, such as individual words and punctuation characters

- ในส่วนนี้ เราจะทำการแบ่งข้อความเป็นโทเค็น (Tokenize) ซึ่งหมายถึงการหั่นข้อความออกเป็นหน่วยย่อยๆ เช่น คำแต่ละคำและเครื่องหมายวรรคตอนต่างๆ

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="800px">

- Load raw text we want to work with
- [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) is a public domain short story

- โหลดข้อความดิบ (Raw text) ที่เราต้องการนำมาใช้งาน
- เรื่องสั้น The Verdict โดย Edith Wharton เป็นงานเขียนที่อยู่ในสาธารณสมบัติ (Public domain) ไม่มีลิขสิทธิ์

In [ ]:
'''
import os  # นำเข้าโมดูล os สำหรับจัดการไฟล์และตรวจสอบตำแหน่งไฟล์ในระบบปฏิบัติการ
import requests  # นำเข้าไลบรารี requests สำหรับส่งคำขอ (HTTP Request) เพื่อดาวน์โหลดข้อมูลจากอินเทอร์เน็ต

# ตรวจสอบว่าในโฟลเดอร์ปัจจุบัน "ยังไม่มี" ไฟล์ชื่อ "the-verdict.txt" อยู่ใช่หรือไม่
if not os.path.exists("the-verdict.txt"):
    # กำหนด URL ต้นทางของไฟล์ข้อความบน GitHub (เขียนต่อสตริงหลายบรรทัดเพื่อให้อ่านง่าย)
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    # กำหนดชื่อไฟล์ปลายทางที่จะบันทึกไว้ในเครื่อง
    file_path = "the-verdict.txt"

    # ส่งคำขอไปดาวน์โหลดไฟล์จาก URL โดยตั้งเวลารอสูงสุด (Timeout) ไว้ที่ 30 วินาที
    response = requests.get(url, timeout=30)
    
    # ตรวจสอบสถานะการดาวน์โหลด หากเกิดข้อผิดพลาด (เช่น ลิงก์เสีย 404 หรือเน็ตหลุด) ให้แจ้งเตือน Error ทันที
    response.raise_for_status()
    
    # เปิดหรือสร้างไฟล์ใหม่ในโหมดเขียนข้อมูลแบบไบนารี ("wb" = Write Binary) เพื่อรักษาโครงสร้างอักขระให้ครบถ้วน
    with open(file_path, "wb") as f:
        # นำเนื้อหาข้อมูลดิบ (Bytes) ที่ดาวน์โหลดมาได้ เขียนบันทึกลงในไฟล์
        f.write(response.content)
'''

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt"
    )
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

<br>

---

<br>

#### Troubleshooting SSL certificate errors

- Some readers reported seeing ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED` when running `urllib.request.urlretrieve` in VSCode or Jupyter. 
- This usually means Python's certificate bundle is outdated.


**Fixes**

- Use Python ≥ 3.9; you can check your Python version by executing the following code:
```python
import sys
print(sys.version)
```
- Upgrade the cert bundle:
  - pip: `pip install --upgrade certifi`
  - uv: `uv pip install --upgrade certifi`
- Restart the Jupyter kernel after upgrading.
- If you still encounter an `ssl.SSLCertVerificationError` when executing the previous code cell, please see the discussion at [more information here on GitHub](https://github.com/rasbt/LLMs-from-scratch/pull/403)

<br>

---

<br>

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- The goal is to tokenize and embed this text for an LLM
- Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above
- The following regular expression will split on whitespaces

- เป้าหมายของเราคือการแปลงข้อความนี้เป็นโทเค็น (Tokenize) และทำ Embedding เพื่อนำไปใช้กับ LLM
- เรามาลองพัฒนา Tokenizer อย่างง่ายโดยเริ่มจากข้อความตัวอย่างสั้นๆ กันก่อน เพื่อที่จะได้นำไปประยุกต์ใช้กับข้อความเรื่องสั้นด้านบนในภายหลัง
- นิพจน์ปรกติ (Regular expression) ต่อไปนี้ จะทำหน้าที่แยกข้อความออกจากกันตามช่องว่าง (Whitespaces)

In [12]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well
*<br><small>เราไม่ได้ต้องการแยกข้อความออกจากกันตามช่องว่าง (Whitespaces) เพียงอย่างเดียว แต่ต้องการแยกตามเครื่องหมายจุลภาค (Commas) และเครื่องหมายมหัพภาค/จุด (Periods) ด้วย ดังนั้นเรามาปรับแก้ Regular expression ให้ทำแบบนั้นได้เช่นกัน*

In [13]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- As we can see, this creates empty strings, let's remove them
*<br><small>อย่างที่เราเห็น วิธีนี้จะทำให้เกิดสตริงว่างเปล่า (Empty strings) ขึ้นมา เรามาลบพวกมันทิ้งกันครับ*

In [14]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on
<br><small>*ผลลัพธ์นี้ดูค่อนข้างดีแล้ว แต่เรามาจัดการกับเครื่องหมายวรรคตอนประเภทอื่นๆ ด้วยดีกว่า เช่น เครื่องหมายมหัพภาค/จุด (Periods) เครื่องหมายปรัศนี/คำถาม (Question marks) และอื่นๆ*

In [15]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- This is pretty good, and we are now ready to apply this tokenization to the raw text
<br>*<small>ผลลัพธ์นี้ค่อนข้างดีทีเดียว และตอนนี้เราก็พร้อมที่จะนำวิธีการแบ่งโทเค็น (Tokenization) นี้ไปใช้กับข้อความดิบ (Raw text)*

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="800px">

In [16]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

NameError: name 'raw_text' is not defined

- Let's calculate the total number of tokens

In [9]:
print(len(preprocessed))

4690


&nbsp;
## 2.3 Converting tokens into token IDs

Converting tokens into token IDs คือกระบวนการเปลี่ยนชิ้นส่วนของข้อความ (Token) ให้กลายเป็น รหัสตัวเลขจำนวนเต็ม (Token ID) เพื่อส่งต่อให้โมเดล AI หรือ Large Language Model (LLM) นำไปคำนวณและประมวลผลได้
เนื่องจากคอมพิวเตอร์และโครงข่ายประสาทเทียม (Neural Network) ไม่สามารถเข้าใจตัวอักษรของมนุษย์ได้โดยตรง จึงต้องแปลงข้อมูลทุกอย่างให้อยู่ในรูปของตัวเลขก่อนเสมอ

- Next, we convert the text tokens into token IDs that we can process via embedding layers later
- *<small>ขั้นตอนถัดไป เราจะแปลงโทเค็นข้อความ (Text tokens) ให้กลายเป็นรหัสโทเค็น (Token IDs) เพื่อที่เราจะสามารถนำไปประมวลผลผ่านชั้น Embedding (Embedding layers) ในภายหลังได้*

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="800px">

- From these tokens, we can now build a vocabulary that consists of all the unique tokens
- *<small>จากโทเค็นเหล่านี้ ตอนนี้เราสามารถสร้างพจนานุกรมคำศัพท์ (Vocabulary) ที่รวบรวมโทเค็นที่ไม่ซ้ำกันทั้งหมด (Unique tokens)*

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

NameError: name 'preprocessed' is not defined

In [11]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- Below are the first 50 entries in this vocabulary:

In [12]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Below, we illustrate the tokenization of a short sample text using a small vocabulary:
- *<small>ด้านล่างนี้ เราจะแสดงตัวอย่างการแบ่งโทเค็น (Tokenization) ของข้อความสั้นๆ โดยใช้พจนานุกรมคำศัพท์ (Vocabulary) ขนาดเล็ก :*

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="800px">

- Putting it now all together into a tokenizer class

- ตอนนี้เรามารวบรวมทุกส่วนเข้าด้วยกันเพื่อสร้างเป็นคลาส Tokenizer (Tokenizer class) 

In [ ]:
'''
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
'''

In [ ]:
# สร้างคลาส SimpleTokenizerV1 สำหรับแปลงข้อความเป็นตัวเลข (Encode) และแปลงตัวเลขกลับเป็นข้อความ (Decode)
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # เก็บพจนานุกรมที่แปลงจาก "คำศัพท์ (String) -> รหัสตัวเลข (Integer ID)"
        self.str_to_int = vocab
        
        # สร้างพจนานุกรมย้อนกลับ โดยสลับคู่จาก "รหัสตัวเลข (Integer ID) -> คำศัพท์ (String)" เพื่อใช้ตอนถอดรหัส (Decode)
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    # เมทอดสำหรับแปลงข้อความดิบ (Text) ให้กลายเป็นชุดรหัสตัวเลข (Token IDs)
    def encode(self, text):
        # ใช้ Regular Expression (re.split) หั่นข้อความออกตามเครื่องหมายวรรคตอน (เช่น , . ? !) ขีดคู่ (--) และช่องว่าง (\s)
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        # ลบช่องว่างส่วนเกินหัวท้ายของแต่ละคำ (.strip()) และคัดกรองเอาเฉพาะคำที่ไม่ใช่ค่าว่างเก็บไว้ในลิสต์
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        
        # นำคำที่ผ่านการตัดแบ่งแล้วแต่ละคำ ไปเทียบหาค่าตัวเลข (ID) ในพจนานุกรม str_to_int
        ids = [self.str_to_int[s] for s in preprocessed]
        
        # ส่งคืนลิสต์ของรหัสตัวเลขทั้งหมด
        return ids
        
    # เมทอดสำหรับแปลงชุดรหัสตัวเลข (Token IDs) กลับคืนมาเป็นประโยคข้อความที่อ่านรู้เรื่อง
    def decode(self, ids):
        # ดึงคำศัพท์ที่ตรงกับแต่ละรหัสตัวเลขจาก int_to_str แล้วนำมาเชื่อมต่อกันโดยคั่นด้วยช่องว่าง 1 เคาะ (" ")
        text = " ".join([self.int_to_str[i] for i in ids])
        
        # ลบช่องว่างส่วนเกินที่อยู่หน้าเครื่องหมายวรรคตอนออก (เช่น เปลี่ยน "Hello , world !" ให้เป็น "Hello, world!")
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        
        # ส่งคืนข้อความที่จัดรูปแบบสมบูรณ์แล้ว
        return text


- The `encode` function turns text into token IDs
- The `decode` function turns token IDs back into text

- ฟังก์ชัน encode ทำหน้าที่แปลงข้อความ (Text) ให้กลายเป็นรหัสโทเค็น (Token IDs)
- ฟังก์ชัน decode ทำหน้าที่แปลงรหัสโทเค็น (Token IDs) กลับคืนมาเป็นข้อความ (Text)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="850px">

- We can use the tokenizer to encode (that is, tokenize) texts into integers
- These integers can then be embedded (later) as input of/for the LLM

- เราสามารถใช้ Tokenizer เพื่อเข้ารหัส (หรือก็คือการแบ่งโทเค็น) ข้อความให้กลายเป็นตัวเลขจำนวนเต็ม (Integers)
- จากนั้นตัวเลขจำนวนเต็มเหล่านี้จะถูกนำไปทำ Embedding (ในภายหลัง) เพื่อใช้เป็นข้อมูลนำเข้า (Input) สำหรับโมเดล LLM

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- We can decode the integers back into text

- เราสามารถถอดรหัส (Decode) ตัวเลขจำนวนเต็มเหล่านั้นให้กลับคืนมาเป็นข้อความ

In [15]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [1]:
tokenizer.decode(tokenizer.encode(text))

NameError: name 'tokenizer' is not defined

&nbsp;
## 2.4 Adding special context tokens *การเพิ่มโทเค็นพิเศษสำหรับระบุบริบท*

- It's useful to add some "special" tokens for unknown words and to denote the end of a text

- การเพิ่มโทเค็น "พิเศษ" (Special tokens) บางตัวเข้าไปนั้นมีประโยชน์มาก เพื่อใช้แทนคำศัพท์ที่ไม่รู้จัก (Unknown words) และใช้ระบุจุดสิ้นสุดของข้อความ (End of a text)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="850px">

- Some tokenizers use special tokens to help the LLM with additional context
- Some of these special tokens are
  - `[BOS]` (beginning of sequence) marks the beginning of text
  - `[EOS]` (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)
  - `[PAD]` (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)
- `[UNK]` to represent words that are not included in the vocabulary

- Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity
- The `<|endoftext|>` is analogous to the `[EOS]` token mentioned above
- GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)
- GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding (BPE) tokenizer, which breaks down words into subword units which we will discuss in a later section



----

- Tokenizer บางตัวมีการใช้โทเค็นพิเศษ (Special tokens) เพื่อช่วยเพิ่มบริบท (Context) ให้กับโมเดล LLM
- ตัวอย่างของโทเค็นพิเศษเหล่านี้ ได้แก่
	- `[BOS]` (beginning of sequence): ใช้ระบุจุดเริ่มต้นของข้อความ
	- `[EOS]` (end of sequence): ใช้ระบุจุดสิ้นสุดของข้อความ (โดยปกติมักใช้เพื่อเชื่อมต่อข้อความหลายๆ เรื่องที่ไม่เกี่ยวข้องกันเข้าด้วยกัน เช่น บทความ Wikipedia สองเรื่องที่ต่างกัน หรือหนังสือสองเล่มที่ต่างกัน เป็นต้น)
	- `[PAD]` (padding): ใช้ในกรณีที่เราเทรน LLM ด้วยขนาด Batch ที่มากกว่า 1 (ซึ่งอาจมีข้อความหลายชุดที่มีความยาวไม่เท่ากัน เราจึงใช้โทเค็น Padding เติมเข้าไปในข้อความที่สั้นกว่าให้ยาวเท่ากับข้อความที่ยาวที่สุด เพื่อให้ทุกข้อความมีความยาวเท่ากันทั้งหมด)
- `[UNK]`: ใช้แทนคำศัพท์ที่ไม่มีอยู่ในพจนานุกรมคำศัพท์ (Vocabulary)
- ข้อสังเกตคือโมเดล GPT-2 ไม่จำเป็นต้องใช้โทเค็นใดๆ ที่กล่าวมาข้างต้นเลย แต่จะใช้เพียงโทเค็น <|endoftext|> เพียงตัวเดียวเพื่อลดความซับซ้อน
- โทเค็น <|endoftext|> มีหน้าที่คล้ายคลึงกับโทเค็น [EOS] ที่กล่าวถึงไปข้างต้น
- นอกจากนี้ GPT ยังใช้ <|endoftext|> สำหรับการทำ Padding ด้วย (เนื่องจากโดยปกติเราจะใช้ Mask เวลาเทรนข้อมูลแบบ Batch อยู่แล้ว โมเดลจึงไม่ได้ให้ความสนใจกับโทเค็นส่วนที่เติมเข้ามาตั้งแต่แรก ดังนั้นจึงไม่สำคัญว่าโทเค็นที่ใช้เติมจะเป็นตัวอะไร)
- โมเดล GPT-2 ไม่ได้ใช้โทเค็น <UNK> สำหรับคำศัพท์ที่อยู่นอกพจนานุกรม แต่จะใช้ตัวแบ่งคำแบบ Byte-Pair Encoding (BPE) แทน ซึ่งจะทำหน้าที่หั่นคำศัพท์ออกเป็นหน่วยคำย่อย (Subword units) โดยเราจะพูดถึงเรื่องนี้ในหัวข้อถัดไป


- We use the `<|endoftext|>` tokens between two independent sources of text:
- เราใช้โทเค็น <|endoftext|> คั่นระหว่างแหล่งที่มาของข้อความสองชุดที่เป็นอิสระต่อกัน (ไม่เกี่ยวข้องกัน):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="850px">

- Let's see what happens if we tokenize the following text:
---
- เรามาดูกันว่าจะเกิดอะไรขึ้น หากเราทำการแบ่งโทเค็น (Tokenize) ข้อความต่อไปนี้:

In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- The above produces an error because the word "Hello" is not contained in the vocabulary
- To deal with such cases, we can add special tokens like `"<|unk|>"` to the vocabulary to represent unknown words
- Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)

---

- โค้ดด้านบนเกิดข้อผิดพลาด (Error) เนื่องจากคำว่า "Hello" ไม่มีอยู่ในพจนานุกรมคำศัพท์ (Vocabulary)
- เพื่อรับมือกับกรณีแบบนี้ เราสามารถเพิ่มโทเค็นพิเศษ (Special tokens) อย่าง "<|unk|>" เข้าไปในพจนานุกรม เพื่อใช้แทนคำศัพท์ที่ไม่รู้จักได้
- และไหนๆ เราก็จะขยายพจนานุกรมคำศัพท์อยู่แล้ว เรามาเพิ่มโทเค็นอีกตัวหนึ่งชื่อว่า "<|endoftext|>" เข้าไปด้วยเลย ซึ่งโทเค็นนี้ถูกใช้ในการเทรนโมเดล GPT-2 เพื่อระบุจุดสิ้นสุดของข้อความ (และยังใช้คั่นระหว่างข้อความที่นำมาเชื่อมต่อกัน เช่น กรณีที่ชุดข้อมูลเทรนของเราประกอบด้วยบทความหรือหนังสือหลายๆ เล่ม เป็นต้น)

In [ ]:
'''
# 1. คัดกรองเอาเฉพาะคำศัพท์ที่ไม่ซ้ำกันด้วย set() แปลงกลับเป็น list() แล้วเรียงลำดับตามตัวอักษรด้วย sorted()
all_tokens = sorted(list(set(preprocessed)))

# 2. เพิ่มโทเค็นพิเศษ 2 ตัวเข้าไปต่อท้ายรายการคำศัพท์ ได้แก่:
#    - "<|endoftext|>" ใช้ระบุจุดสิ้นสุดของข้อความหรือคั่นระหว่างข้อความแต่ละเรื่อง
#    - "<|unk|>" ใช้แทนคำศัพท์ที่ไม่รู้จัก (ไม่มีอยู่ในพจนานุกรม)
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

# 3. สร้างพจนานุกรมคำศัพท์ (Vocabulary) ที่จับคู่ระหว่าง "คำศัพท์ (token)" กับ "รหัสตัวเลขจำนวนเต็ม (integer)"
#    โดยใช้ enumerate() เพื่อรันหมายเลขลำดับตั้งแต่ 0, 1, 2, ... ไปจนครบทุกคำ
vocab = {token: integer for integer, token in enumerate(all_tokens)}
'''

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}


In [19]:
len(vocab.items())

1132

In [20]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token
- เรายังจำเป็นต้องปรับแก้ตัวแบ่งโทเค็น (Tokenizer) ให้สอดคล้องกันด้วย เพื่อให้มันรู้ว่าควรจะนำโทเค็น `<unk>` ตัวใหม่นี้ไปใช้เมื่อไหร่และใช้อย่างไร

คลาส SimpleTokenizerV2 คือเวอร์ชันอัปเกรดของตัวแบ่งคำ ที่เพิ่มความสามารถในการ รับมือกับคำศัพท์ใหม่ที่ไม่มีอยู่ในพจนานุกรม (Unknown words) เพื่อไม่ให้โปรแกรมเกิดข้อผิดพลาด KeyError เหมือนในเวอร์ชันแรกครับ

In [17]:
'''
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # เก็บพจนานุกรมแปลง "คำศัพท์ -> รหัสตัวเลข"
        self.str_to_int = vocab
        # สร้างพจนานุกรมย้อนกลับแปลง "รหัสตัวเลข -> คำศัพท์" สำหรับใช้ตอนถอดรหัส
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        # 1. หั่นข้อความออกตามเครื่องหมายวรรคตอน ขีดคู่ (--) และช่องว่าง
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        
        # 2. ลบช่องว่างหัวท้ายของแต่ละคำ และตัดสตริงว่างเปล่าทิ้ง
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        
        # 3. [ส่วนที่อัปเกรดใหม่] ตรวจสอบคำศัพท์ทีละคำว่ามีอยู่ในพจนานุกรม (self.str_to_int) หรือไม่
        #    - ถ้ามี: ให้ใช้คำศัพท์เดิม (item)
        #    - ถ้าไม่มี: ให้เปลี่ยนคำนั้นเป็นโทเค็นพิเศษ "<|unk|>" ทันที เพื่อป้องกัน Error
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        # 4. แปลงคำศัพท์ทั้งหมดให้เป็นรหัสตัวเลข (Token IDs) แล้วส่งค่ากลับ
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        # 1. แปลงรหัสตัวเลขกลับเป็นคำศัพท์ แล้วนำมาต่อกันโดยคั่นด้วยช่องว่าง 1 เคาะ
        text = " ".join([self.int_to_str[i] for i in ids])
        
        # 2. ลบช่องว่างส่วนเกินที่อยู่หน้าเครื่องหมายวรรคตอนออก (เพิ่มเครื่องหมาย : และ ; เข้ามาจาก V1)
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text
'''


'\nclass SimpleTokenizerV2:\n    def __init__(self, vocab):\n        # เก็บพจนานุกรมแปลง "คำศัพท์ -> รหัสตัวเลข"\n        self.str_to_int = vocab\n        # สร้างพจนานุกรมย้อนกลับแปลง "รหัสตัวเลข -> คำศัพท์" สำหรับใช้ตอนถอดรหัส\n        self.int_to_str = {i: s for s, i in vocab.items()}\n\n    def encode(self, text):\n        # 1. หั่นข้อความออกตามเครื่องหมายวรรคตอน ขีดคู่ (--) และช่องว่าง\n        preprocessed = re.split(r\'([,.:;?_!"()\']|--|\\s)\', text)\n\n        # 2. ลบช่องว่างหัวท้ายของแต่ละคำ และตัดสตริงว่างเปล่าทิ้ง\n        preprocessed = [item.strip() for item in preprocessed if item.strip()]\n\n        # 3. [ส่วนที่อัปเกรดใหม่] ตรวจสอบคำศัพท์ทีละคำว่ามีอยู่ในพจนานุกรม (self.str_to_int) หรือไม่\n        #    - ถ้ามี: ให้ใช้คำศัพท์เดิม (item)\n        #    - ถ้าไม่มี: ให้เปลี่ยนคำนั้นเป็นโทเค็นพิเศษ "<|unk|>" ทันที เพื่อป้องกัน Error\n        preprocessed = [\n            item if item in self.str_to_int \n            else "<|unk|>" for item in preprocessed\n        ]\n\n     


จุดที่อัปเกรดขึ้นมาจาก `SimpleTokenizerV1`

- ป้องกันโปรแกรมพังเมื่อเจอคำใหม่: ในเมทอด encode() มีการเพิ่มเงื่อนไข item if item in self.str_to_int else "<|unk|>" หากโมเดลเจอคำที่ไม่เคยเห็นมาก่อน จะถูกแทนที่ด้วยรหัสของ <|unk|> ทันที
- รองรับเครื่องหมายวรรคตอนครบขึ้นใน `decode():` มีการเพิ่มเครื่องหมายทวิภาค `(:)` และอัฒภาค `(;)` เข้าไปในคำสั่ง `re.sub` เพื่อลบช่องว่างหน้าเครื่องหมายเหล่านี้ให้ประโยคสวยงามสมบูรณ์ครับ
ขั้นตอนถัดไปในหนังสือจะเป็นการทดสอบใช้งาน `SimpleTokenizerV2` กับประโยคที่มีคำศัพท์ใหม่และมีตัวคั่น `<|endoftext|>`

In [18]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Let's try to tokenize text with the modified tokenizer :
<br><small>เรามาลองแบ่งโทเค็น (Tokenize) ข้อความด้วยตัวแบ่งโทเค็นที่ปรับแก้ใหม่กัน :

In [22]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [23]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [24]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

&nbsp;
## 2.5 BytePair encoding 

*2.5 การเข้ารหัสแบบ BytePair (BytePair encoding หรือ BPE)*

- GPT-2 used BytePair encoding (BPE) as its tokenizer
- it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words
- For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges
- The original BPE tokenizer can be found here: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- In this chapter, we are using the BPE tokenizer from OpenAI's open-source [tiktoken](https://github.com/openai/tiktoken) library, which implements its core algorithms in Rust to improve computational performance
- I created a notebook in the [./bytepair_encoder](../02_bonus_bytepair-encoder) that compares these two implementations side-by-side (tiktoken was about 5x faster on the sample text)

---

- โมเดล GPT-2 ใช้การเข้ารหัสแบบ BytePair encoding (BPE) เป็นตัวแบ่งโทเค็น (Tokenizer)
- วิธีนี้ช่วยให้โมเดลสามารถหั่นคำศัพท์ที่ไม่มีอยู่ในพจนานุกรมที่กำหนดไว้ล่วงหน้า ให้กลายเป็นหน่วยคำย่อย (Subword units) ที่เล็กลง หรือแม้กระทั่งแยกเป็นตัวอักษรเดี่ยวๆ ได้ ซึ่งทำให้โมเดลสามารถจัดการกับคำศัพท์นอกพจนานุกรม (Out-of-vocabulary words) ได้อย่างมีประสิทธิภาพ
- ตัวอย่างเช่น หากพจนานุกรมของ GPT-2 ไม่มีคำว่า "unfamiliarword" มันอาจจะแบ่งโทเค็นคำนี้ออกเป็น ["unfam", "iliar", "word"] หรือแบ่งเป็นหน่วยคำย่อยแบบอื่นๆ ขึ้นอยู่กับกฎการรวมคำ (BPE merges) ที่โมเดลได้เรียนรู้มา
- โค้ดต้นฉบับของตัวแบ่งโทเค็นแบบ BPE สามารถดูได้ที่นี่: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- ในบทนี้ เราจะใช้ตัวแบ่งโทเค็นแบบ BPE จากไลบรารีโอเพนซอร์ส [tiktoken](https://github.com/openai/tiktoken) ของ OpenAI ซึ่งพัฒนาอัลกอริทึมหลักด้วยภาษา Rust เพื่อเพิ่มประสิทธิภาพในการคำนวณให้รวดเร็วขึ้น
- ผู้เขียนได้สร้างสมุดบันทึกโค้ด (Notebook) ไว้ใน [./bytepair_encoder](../02_bonus_bytepair-encoder) เพื่อเปรียบเทียบการทำงานของทั้งสองรูปแบบแบบเคียงข้างกัน (ซึ่ง tiktoken ทำงานเร็วกว่าประมาณ 5 เท่ากับข้อความตัวอย่าง)

In [25]:
# pip install tiktoken

In [26]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.7.0


In [27]:
tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [29]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizers break down unknown words into subwords and individual characters:
- ตัวแบ่งโทเค็นแบบ BPE จะหั่นคำศัพท์ที่ไม่รู้จักออกเป็นหน่วยคำย่อย (Subwords) และตัวอักษรเดี่ยวๆ (Individual characters):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="850px">

## 2.6 Data sampling with a sliding window

*2.6 การสุ่มตัวอย่างข้อมูลด้วย sliding window*

- We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:
---
- เราเทรนโมเดล LLM ให้สร้างข้อความออกมาทีละคำ ดังนั้นเราจึงต้องเตรียมข้อมูลสำหรับฝึกสอนให้สอดคล้องกัน โดยให้คำถัดไปในลำดับ (Sequence) ทำหน้าที่เป็นคำตอบเป้าหมาย (Target) ที่โมเดลต้องทำนาย

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="850px">

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- For each text chunk, we want the inputs and targets
- Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right
---
- สำหรับข้อความแต่ละชุดย่อย (Text chunk) เราต้องการทั้งส่วนที่เป็นข้อมูลนำเข้า (Inputs) และคำตอบเป้าหมาย (Targets)
- เนื่องจากเราต้องการให้โมเดลทำนายคำถัดไป ดังนั้นข้อมูลในส่วนของคำตอบเป้าหมาย (Targets) จึงคือข้อมูลนำเข้า (Inputs) ที่ถูกเลื่อนตำแหน่งไปทางขวา 1 ตำแหน่ง

In [31]:
enc_sample = enc_text[50:]

In [32]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- One by one, the prediction would look like as follows:
<br>*เมื่อแสดงการทำนายทีละขั้นตอน ผลลัพธ์จะมีหน้าตาดังต่อไปนี้ :*

In [ ]:
'''
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)
'''

'''
คำสั่ง for i in range(1, context_size+1) เป็นการสร้างลูปเพื่อกำหนดขนาดความยาวของข้อความบริบท (Context) ที่จะป้อนให้โมเดล ตั้งแต่ 1 โทเค็น ขยายเพิ่มขึ้นทีละ 1 ไปจนครบ จำนวน context_size โทเค็น ครับ

การทำงานของแต่ละส่วน

• range(start, stop): ในภาษา Python ฟังก์ชัน range จะเริ่มนับตั้งแต่ตัวเลข start ไปจนถึงตัวเลขก่อนหน้า stop (ไม่รวมตัว stop เอง)
• ทำไมต้องเริ่มที่ 1: เพื่อให้ในรอบแรกของการทำนาย โมเดลมีข้อมูลบริบทตั้งต้นอย่างน้อย 1 คำ (enc_sample[:1]) สำหรับใช้ทำนายคำที่ 2 (enc_sample[1]) ถ้าเริ่มที่ 0 ข้อมูลตั้งต้นจะว่างเปล่า
• ทำไมต้องเป็น context_size + 1: เพราะฟังก์ชัน range จะหยุดทำงานก่อนถึงเลขตัวสุดท้าย 1 ก้าว การบวกเพิ่มไปอีก 1 จึงทำให้ตัวแปร i วิ่งไปจนถึงค่า context_size แบบพอดีเป๊ะ

ตัวอย่างการทำงานจริง

สมมติว่าตั้งค่า context_size = 4 โค้ดนี้จะกลายเป็น range(1, 5) ซึ่งตัวแปร i จะมีค่าดังนี้ครับ:
• รอบที่ 1 (i = 1): ดึงข้อมูล [:1] (1 คำแรก) เพื่อทำนายคำที่ [1] (คำที่ 2)
• รอบที่ 2 (i = 2): ดึงข้อมูล [:2] (2 คำแรก) เพื่อทำนายคำที่ [2] (คำที่ 3)
• รอบที่ 3 (i = 3): ดึงข้อมูล [:3] (3 คำแรก) เพื่อทำนายคำที่ [3] (คำที่ 4)
• รอบที่ 4 (i = 4): ดึงข้อมูล [:4] (4 คำแรก) เพื่อทำนายคำที่ [4] (คำที่ 5) แล้วจบการทำงาน

'''
# วนลูปเพิ่มขนาดของบริบท (Context) ทีละ 1 โทเค็น ตั้งแต่ 1 ไปจนถึงขนาดสูงสุดที่กำหนดไว้ (context_size)
for i in range(1, context_size+1):
    # ดึงรหัสโทเค็นตั้งแต่อันแรกจนถึงตำแหน่งที่ i-1 เพื่อใช้เป็นข้อมูลบริบทนำเข้า (Input context)
    context = enc_sample[:i]
    
    # ดึงรหัสโทเค็นที่ตำแหน่ง i ซึ่งอยู่ถัดไป 1 ก้าว เพื่อใช้เป็นคำตอบเป้าหมายที่โมเดลต้องทำนาย (Target)
    desired = enc_sample[i]

    # แสดงผลคู่ของข้อมูลบริบทนำเข้า (ซ้าย) และคำตอบที่ถูกต้องถัดไป (ขวา)
    print(context, "---->", desired)


[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [1]:
'''
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
'''

'\nfor i in range(1, context_size+1):\n    context = enc_sample[:i]\n    desired = enc_sample[i]\n\n    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))\n'

### 🔍 อธิบายโค้ด
- context = enc_sample[:i] (เอาตั้งแต่ตัวแรก จนถึง "ก่อนหน้า" ตำแหน่ง i)
	- เครื่องหมายโคลอน : หมายถึงการดึงข้อมูลเป็น ช่วง (Slice) ตั้งแต่เริ่มต้น (0) ไปจนถึงตำแหน่ง i - 1
	- ข้อมูลที่ได้ออกมาจะเป็น List ของรหัสโทเค็น เพื่อใช้เป็นข้อความบริบทตั้งต้นป้อนให้โมเดลอ่าน
- desired = enc_sample[i] (เอาเฉพาะตัวที่อยู่ "ตรงตำแหน่ง i พอดี")
	- การระบุ [i] โดยไม่มีโคลอน หมายถึงการเจาะจงหยิบข้อมูล เพียงตัวเดียว ที่ตำแหน่ง i
	- เนื่องจากใน Python เริ่มนับตำแหน่งแรกที่ 0 ตำแหน่ง i จึงเป็นตัวที่อยู่ถัดจากช่วง [:i] มา 1 ก้าวพอดี ซึ่งก็คือคำถัดไปที่โมเดลต้องทำนายให้ถูก
### 💡ตัวอย่างให้เห็นภาพชัดเจน
สมมติว่าลิสต์ enc_sample เก็บคำศัพท์ 5 คำ คือ ["แมว", "กิน", "ปลา", "ทู", "อร่อย"] (ซึ่งมีตำแหน่งดัชนีหรือ Index คือ 0, 1, 2, 3, 4)
หากในรอบนั้นตัวแปร i = 3 โค้ดจะทำงานดังนี้ครับ:
- enc_sample[:3] → ดึงตำแหน่งที่ 0, 1, 2 จะได้ ["แมว", "กิน", "ปลา"] (เป็นกลุ่มคำใบ้ตั้งต้น)
- enc_sample[3] → ดึงตำแหน่งที่ 3 ตัวเดียว จะได้ "ทู" (เป็นคำถัดไปที่โมเดลต้องทาย)

In [7]:
# วนลูปเพิ่มขนาดของบริบท (Context) ทีละ 1 โทเค็น ตั้งแต่ 1 ไปจนถึงขนาดสูงสุดที่กำหนดไว้ (context_size)
for i in range(1, context_size+1):
    # ดึงรหัสโทเค็นตั้งแต่อันแรกจนถึงตำแหน่งที่ i-1 เพื่อใช้เป็นข้อมูลบริบทนำเข้า (Input context)
    context = enc_sample[:i]
    
    # ดึงรหัสโทเค็นที่ตำแหน่ง i ซึ่งอยู่ถัดไป 1 ก้าว เพื่อใช้เป็นคำตอบเป้าหมายที่โมเดลต้องทำนาย (Target)
    desired = enc_sample[i]

    # แปลงรหัสโทเค็นกลับเป็นข้อความที่อ่านรู้เรื่องด้วย tokenizer.decode() แล้วพิมพ์แสดงผลคู่คำทำนาย
    # หมายเหตุ: ต้องครอบ desired ด้วยเครื่องหมายก้ามปู [desired] เพราะเมทอด decode() รับข้อมูลเข้าเป็น List ของตัวเลขเท่านั้น
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

NameError: name 'context_size' is not defined

### 🔍 อธิบายการทำงานทีละบรรทัด
- for i in range(1, context_size+1):
	- สร้างลูปเพื่อขยายความยาวของข้อความบริบท (Context) ทีละ 1 โทเค็น โดยเริ่มจาก 1 คำ ไปจนครบจำนวน context_size ที่กำหนดไว้
- context = enc_sample[:i]
	- ดึงรหัสโทเค็นตั้งแต่อันแรกสุดไปจนถึงตำแหน่งก่อนหน้า i เพื่อใช้เป็น ข้อความตั้งต้น (Input) ให้โมเดลอ่าน
- desired = enc_sample[i]
	- ดึงรหัสโทเค็นที่ตำแหน่ง i (ซึ่งก็คือคำที่อยู่ถัดไป 1 ก้าวพอดี) เพื่อใช้เป็น คำตอบเฉลย (Target) ที่โมเดลต้องทายให้ถูก
- print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
	- นำรหัสตัวเลขทั้งสองส่วนมาแปลงกลับเป็นข้อความด้วยเมทอด .decode() แล้วพิมพ์แสดงผลคู่กัน
	- จุดสังเกตสำคัญ: ตัวแปร desired ต้องถูกครอบด้วยวงเล็บก้ามปู [desired] เพราะ desired เป็นตัวเลขเดี่ยวๆ (Integer) แต่ฟังก์ชัน tokenizer.decode() บังคับให้ส่งข้อมูลเข้าไปเป็น รายการ (List) เท่านั้น

### 💡 ตัวอย่างผลลัพธ์ที่จะแสดงบนหน้าจอ
สมมติว่าข้อความตัวอย่างคือ "In the heart of the city" และตั้งค่า context_size = 4 ผลลัพธ์ที่พิมพ์ออกมาจะเป็นแบบนี้ครับ:
- In ----> the
- In the ----> heart
- In the heart ----> of
- In the heart of ----> the

---

- We will take care of the next-word prediction in a later chapter after we covered the attention mechanism
<br><sub>- เราจะจัดการเรื่องการทำนายคำถัดไป (Next-word prediction) ในบทถัดๆ ไป หลังจากที่เราได้เรียนรู้เรื่องกลไกความสนใจ (Attention mechanism) เรียบร้อยแล้ว</sub>
<br><br>

- For now, we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one
<br><small>- สำหรับตอนนี้ เราจะสร้างตัวโหลดข้อมูล (Data loader) อย่างง่าย ที่ทำหน้าที่วนลูปดึงข้อมูลจากชุดข้อมูลนำเข้า และส่งคืนค่าข้อมูลนำเข้า (Inputs) กับคำตอบเป้าหมาย (Targets) ที่ถูกเลื่อนตำแหน่งไป 1 ตำแหน่ง</small>

- Install and import PyTorch (see Appendix A for installation tips)

In [3]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.8.0+cpu


- We use a sliding window approach, changing the position by +1:
<br><sub>*เราใช้วิธีการเลื่อนหน้าต่าง (Sliding window approach) โดยขยับตำแหน่งเพิ่มขึ้นทีละ +1:*

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="850px">

- Create dataset and dataloader that extract chunks from the input text dataset
<br><small>*สร้างชุดข้อมูล (Dataset) และตัวโหลดข้อมูล (Dataloader) ที่ทำหน้าที่ดึงข้อความเป็นชุดย่อยๆ (Chunks) ออกมาจากชุดข้อมูลข้อความนำเข้า*

In [ ]:
'''
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
'''

### 🔍 อธิบายขั้นตอนการตั้งค่าเริ่มต้น (Initialization) ของคลาส GPTDatasetV1 ทีละส่วน

- class GPTDatasetV1(Dataset):
	- สร้างคลาสชุดข้อมูลใหม่โดย สืบทอดคุณสมบัติมาจากคลาส Dataset ของ PyTorch เพื่อให้สามารถนำไปใช้งานร่วมกับ DataLoader ในการแบ่งข้อมูลเป็น Batch ได้อย่างอัตโนมัติ

- def __init__(self, txt, tokenizer, max_length, stride):
	- ฟังก์ชันกำหนดค่าเริ่มต้นเมื่อมีการเรียกสร้างออบเจกต์ โดยรับค่าสำคัญ 4 ตัวเข้ามา:
		- txt: ข้อความดิบทั้งหมดที่ต้องการนำมาเทรน (เช่น เนื้อหานิยายทั้งเรื่อง)
		- tokenizer: ตัวแบ่งคำ (เช่น BPE tokenizer จาก tiktoken) ที่ใช้แปลงตัวอักษรเป็นตัวเลข
		- max_length: ความยาวสูงสุด (จำนวนโทเค็น) ของข้อมูลแต่ละท่อนที่จะป้อนให้โมเดลอ่านใน 1 ครั้ง
		- stride: ระยะก้าวในการเลื่อนหน้าต่าง (Sliding Window) เพื่อตัดข้อมูลท่อนถัดไป

- self.input_ids = [ ] และ self.target_ids = [ ]
	- สร้างลิสต์ว่างเปล่า 2 ชุด เพื่อเตรียมไว้เก็บคู่ข้อมูล:
		- input_ids: เก็บกลุ่มรหัสโทเค็นที่เป็น คำถาม/ข้อความตั้งต้น
		- target_ids: เก็บกลุ่มรหัสโทเค็นที่เป็น คำตอบเฉลย (ซึ่งจะเลื่อนไปทางขวา 1 ตำแหน่ง)

- token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
	- นำข้อความดิบทั้งหมดมาแปลงให้กลายเป็น รายการของรหัสตัวเลข (Token IDs) ทีเดียวทั้งก้อน
	- พารามิเตอร์ allowed_special={"<|endoftext|>"} เป็นการอนุญาตให้ Tokenizer แปลงคำพิเศษอย่าง <|endoftext|> ให้เป็นรหัสโทเค็นพิเศษได้โดยไม่แจ้งเตือน Error

- assert len(token_ids) > max_length, "..."
	- เป็นคำสั่ง ตรวจสอบความปลอดภัย (Safety Check) เพื่อเช็คว่าข้อความทั้งหมดมีจำนวนโทเค็นยาวมากกว่า max_length หรือไม่ (ต้องยาวอย่างน้อย max_length + 1 โทเค็น)
	- เหตุผล: เพราะถ้าสมมติเราตั้ง max_length = 4 เราจะต้องใช้โทเค็น 4 ตัวแรกเป็น Input และต้องมีโทเค็นตัวที่ 5 เหลืออยู่เพื่อเอามาทำเป็น Target ตัวสุดท้าย หากข้อความสั้นเกินไปโปรแกรมจะหยุดทำงานและแจ้งเตือนข้อความ Error ทันที

- for i in range(0, len(token_ids) - max_length, stride):
	- สร้างลูปเพื่อขยับจุดเริ่มต้น (i) ในการตัดข้อความทีละ stride ก้าว
	- ทำไมต้องลบด้วย max_length? เพื่อป้องกันไม่ให้หน้าต่างเลื่อนไปจนตกขอบข้อมูลครับ เพราะในรอบสุดท้าย เรายังต้องการพื้นที่เหลืออีก max_length ตัวสำหรับ input_chunk และอีก 1 ตัวสำหรับ target_chunk

- input_chunk = token_ids[i : i + max_length]
	- ตัดเอาโทเค็นตั้งแต่ตำแหน่ง i ไปจนครบความยาว max_length ตัว เพื่อใช้เป็น ข้อความตั้งต้น (Input)

- target_chunk = token_ids[i + 1 : i + max_length + 1]
	- ตัดเอาโทเค็นที่มีความยาวเท่ากันเป๊ะ แต่ ขยับจุดเริ่มต้นและจุดสิ้นสุดไปทางขวา 1 ตำแหน่ง (+ 1) เพื่อใช้เป็น คำตอบเฉลย (Target) ของแต่ละคำใน input_chunk

- self.input_ids.append(torch.tensor(input_chunk)) (และบรรทัดถัดมา)
	- แปลงลิสต์ตัวเลขธรรมดาของ Python ให้กลายเป็น PyTorch Tensor (โครงสร้างข้อมูลหลักที่ GPU และโมเดลใช้คำนวณ) แล้วนำไปเก็บต่อท้ายลิสต์หลักของคลาส

- `def __len__(self):`
<br>	`return len(self.input_ids)`<br>
	- เป็นฟังก์ชันพิเศษของภาษา Python (เรียกว่า Magic Method หรือ Dunder Method) ที่ทำหน้าที่ บอกจำนวนข้อมูลทั้งหมดที่มีอยู่ในชุดข้อมูล (Dataset)
		- `__len__(self):` เมื่อคุณสร้างคลาสที่สืบทอดมาจาก torch.utils.data.Dataset ของ PyTorch ระบบจะบังคับให้คุณต้องเขียนฟังก์ชันนี้ไว้เสมอ เพื่อให้สามารถใช้คำสั่ง len(dataset) กับออบเจกต์ที่สร้างขึ้น
		- `return len(self.input_ids):` ในขั้นตอน `__init__` เราได้หั่นข้อความยาวๆ ออกเป็นท่อนย่อย (Chunks) แล้วเก็บสะสมไว้ในลิสต์ self.input_ids บรรทัดนี้จึงเป็นการนับจำนวนท่อนข้อมูลทั้งหมดที่มีอยู่ในลิสต์นั้นแล้วส่งค่ากลับออกไป

- def `__getitem__(self, idx):`

### 💡 ตัวอย่างให้เห็นภาพชัดเจน
สมมติว่าข้อความมีรหัสโทเค็น [10, 20, 30, 40, 50, 60] และเราตั้งค่า max_length = 3, stride = 2:

- รอบที่ 1 (i = 0):
	- input_chunk = [10, 20, 30]
	- target_chunk = [20, 30, 40] (เลื่อนขวา 1 ก้าว)

- รอบที่ 2 (i = 2): (ขยับจุดเริ่มเพิ่มทีละ stride = 2)
	- input_chunk = [30, 40, 50] (สังเกตว่าเลข 30 ซ้อนทับกับรอบแรก)
	- target_chunk = [40, 50, 60]

In [ ]:
from torch.utils.data import Dataset, DataLoader


# สร้างคลาสชุดข้อมูลสำหรับ GPT โดยสืบทอดคุณสมบัติมาจาก Dataset ของ PyTorch
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        # สร้างลิสต์ว่างสำหรับเก็บชุดข้อมูลนำเข้า (Inputs) และคำตอบเป้าหมาย (Targets)
        self.input_ids = []
        self.target_ids = []

        # แปลงข้อความดิบทั้งหมดให้เป็นรหัสตัวเลข (Token IDs) โดยอนุญาตให้ประมวลผลโทเค็นพิเศษอย่าง <|endoftext|> ได้
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        
        # ตรวจสอบเงื่อนไขว่าจำนวนโทเค็นทั้งหมดต้องยาวกว่า max_length (อย่างน้อยต้องเท่ากับ max_length + 1) เพื่อให้แบ่งคู่คำตอบได้พอดี
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # ใช้เทคนิคหน้าต่างแบบเลื่อน (Sliding window) เพื่อหั่นข้อความยาวๆ ออกเป็นชุดย่อยที่มีความยาวเท่ากับ max_length
        # โดยตัวแปร stride จะเป็นตัวกำหนดระยะก้าวในการเลื่อนหน้าต่างในแต่ละรอบ (ถ้า stride น้อยกว่า max_length ข้อมูลจะซ้อนทับกัน)
        for i in range(0, len(token_ids) - max_length, stride):
            # ตัดแบ่งข้อมูลนำเข้า (Input) ตั้งแต่ตำแหน่ง i ไปจนครบความยาว max_length
            input_chunk = token_ids[i : i + max_length]
            
            # ตัดแบ่งคำตอบเป้าหมาย (Target) โดยเลื่อนตำแหน่งไปทางขวา 1 ก้าว (เริ่มที่ i + 1) เพื่อใช้เฉลยคำถัดไปของแต่ละคำใน Input
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            
            # แปลงลิสต์ตัวเลขทั้งสองชุดให้เป็น PyTorch Tensor แล้วนำไปเก็บสะสมไว้ในลิสต์หลักของคลาส
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # ส่งคืนจำนวนชุดข้อมูลย่อย (Chunks) ทั้งหมดที่ถูกแบ่งไว้ใน Dataset นี้
        return len(self.input_ids)

    def __getitem__(self, idx):
        # ดึงคู่ข้อมูลนำเข้า (input_ids) และคำตอบเป้าหมาย (target_ids) ที่ตำแหน่ง idx ออกมา 1 ชุดสำหรับการเทรน
        return self.input_ids[idx], self.target_ids[idx]


In [ ]:
'''
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader
'''

In [8]:
# นิยามฟังก์ชันสำหรับสร้าง DataLoader เพื่อเตรียมชุดข้อมูลข้อความสำหรับเทรนโมเดล GPT
def create_dataloader_v1(
    txt,  # ข้อความดิบทั้งหมดที่ต้องการนำมาเตรียมข้อมูล
    batch_size=4,  # จำนวนชุดข้อมูลย่อย (Chunks) ที่จะดึงมาประมวลผลพร้อมกันใน 1 รอบ (Batch)
    max_length=256,  # ความยาวสูงสุด (จำนวนโทเค็น) ของข้อมูลแต่ละชุด
    stride=128,  # ระยะก้าวในการเลื่อนหน้าต่าง (Sliding Window) เพื่อตัดข้อมูลชุดถัดไป
    shuffle=True,  # กำหนดให้สุ่มสลับลำดับข้อมูลในทุกๆ รอบการเทรน (Epoch) เพื่อไม่ให้โมเดลจำลำดับ
    drop_last=True,  # ตัดข้อมูลกลุ่มสุดท้ายทิ้งหากมีจำนวนไม่ครบตามขนาด batch_size (ป้องกัน Error ตอนเทรน)
    num_workers=0,  # จำนวนโปรเซสย่อยของ CPU ที่ใช้ช่วยโหลดข้อมูล (0 หมายถึงใช้โปรเซสหลักตัวเดียว)
):

    # 1. สร้างตัวแบ่งโทเค็น (Tokenizer) แบบ BPE โดยโหลดชุดกฎการเข้ารหัสมาตรฐานของโมเดล "gpt2"
    tokenizer = tiktoken.get_encoding("gpt2")

    # 2. สร้างออบเจกต์ชุดข้อมูล (Dataset) จากคลาส GPTDatasetV1 เพื่อตัดแบ่งข้อความเป็นคู่ Input และ Target
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 3. สร้างตัวโหลดข้อมูล (DataLoader) ของ PyTorch เพื่อจัดการแบ่งข้อมูลเป็น Batch และสุ่มลำดับตามที่ตั้งค่าไว้
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

    # ส่งคืนออบเจกต์ dataloader ที่พร้อมนำไปวนลูปใช้งานในการเทรนโมเดล
    return dataloader


- Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4:
<br><sub>*เรามาลองทดสอบตัวโหลดข้อมูล (Dataloader) ด้วยขนาด Batch เท่ากับ 1 สำหรับโมเดล LLM ที่มีขนาดบริบท (Context size) เท่ากับ 4*

In [6]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

FileNotFoundError: [Errno 2] No such file or directory: 'the-verdict.txt'

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- An example using stride equal to the context length (here: 4) as shown below:
*<br><small>ตัวอย่างการใช้ระยะก้าว (Stride) ที่เท่ากับความยาวบริบท (Context length) (ในที่นี้คือ 4) ดังแสดงด้านล่างนี้:*

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="850px">

- We can also create batched outputs
- Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting
---
- เรายังสามารถสร้างผลลัพธ์แบบเป็นกลุ่มชุดข้อมูล (Batched outputs) ได้อีกด้วย
- ข้อสังเกตคือในส่วนนี้เราเพิ่มค่า ระยะก้าว (Stride) ให้มากขึ้น เพื่อไม่ให้ข้อมูลระหว่างแต่ละ Batch เกิดการซ้อนทับกัน (Overlaps) เนื่องจากการซ้อนทับกันที่มากเกินไปอาจส่งผลให้โมเดลเกิดภาวะจำข้อสอบมากเกินไป (Overfitting)*

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


&nbsp;
## <span style="color: #E2E8F0;">2.7 Creating token embeddings</span>

- The data is already almost ready for an LLM
- But lastly let us embed the tokens in a continuous vector representation using an embedding layer
- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training

---
- ข้อมูลเกือบจะพร้อมสมบูรณ์สำหรับการใช้งานกับโมเดล LLM แล้ว
- แต่ในขั้นตอนสุดท้าย เรามาฝังตัวโทเค็นเหล่านี้ให้อยู่ในรูปแบบการแทนค่าด้วยเวกเตอร์ต่อเนื่อง (Continuous vector representation) โดยใช้ชั้น Embedding (Embedding layer) กันครับ
- โดยปกติแล้ว ชั้น Embedding เหล่านี้จะเป็นส่วนหนึ่งของตัวโมเดล LLM เอง และจะถูกอัปเดต (ฝึกสอน) ไปพร้อมๆ กันในระหว่างการเทรนโมเดล

### <span style="color: #94A3B8;">Note :</span>

<span style="color: #64B5F6;"><b>Token Embeddings</b></span> คือการแปลงรหัสตัวเลขของคำศัพท์ (Token IDs) ให้กลายเป็น ชุดตัวเลขทศนิยมหลายมิติ (Vectors) เพื่อให้โมเดล LLM สามารถนำไปคำนวณและเข้าใจความหมาย รวมถึงความสัมพันธ์ระหว่างคำแต่ละคำได้


<span style="color: #64B5F6;"><b>1. ทำไมเราต้องแปลงเป็น Token Embeddings?</b></span>

- ตัวเลข ID โดดๆ ไม่มีข้อมูลความหมาย: รหัสโทเค็นที่ได้จาก Tokenizer (เช่น 40 แทนคำว่า "I" หรือ 367 แทนคำว่า "had") เป็นเพียงแค่ "ป้ายชื่อ" หรือลำดับที่ในพจนานุกรมเท่านั้น โมเดลไม่สามารถเอาเลข 40 มาบวกลบคูณหารกับ 367 เพื่อหาความหมายได้
- เวกเตอร์ช่วยจับคู่ความสัมพันธ์: เมื่อแปลง 1 คำให้กลายเป็นชุดตัวเลขหลายมิติ (เช่น 256 หรือ 768 ตัวเลขต่อ 1 คำ) ตัวเลขแต่ละหลักจะทำหน้าที่เป็นคุณสมบัติแฝง (เช่น บอกว่าเป็นคำนามหรือกริยา บอกอารมณ์ หรือบริบทการใช้งาน) ทำให้คำที่มีความหมายใกล้เคียงกัน (เช่น "สุนัข" กับ "หมา") มีกลุ่มตัวเลขที่ชี้ไปในทิศทางเดียวกันในพื้นที่มิติสูงครับ

<span style="color: #64B5F6;"><b>2. กลไกการทำงานของ Embedding Layer</b></span>

ในทางปฏิบัติ ชั้น torch.nn.Embedding ของ PyTorch ทำงานเหมือนกับ ตารางค้นหา (Lookup Table) ขนาดใหญ่ครับ:
- ขนาดของตาราง: มีจำนวนแถวเท่ากับจำนวนคำศัพท์ทั้งหมดในพจนานุกรม (vocab_size) และมีจำนวนคอลัมน์เท่ากับขนาดมิติเวกเตอร์ที่เราต้องการ (output_dim)
- การดึงข้อมูล: เมื่อส่งรหัสคำศัพท์ เช่น Token ID = 40 เข้าไป ระบบจะเดินไปที่ แถวที่ 40 ของตาราง แล้วหยิบชุดตัวเลขทศนิยมทั้งหมดในแถวนั้นออกมาแทนที่เลข 40 ทันที
- การเรียนรู้: ในตอนเริ่มต้น ตัวเลขในตารางนี้จะถูกสุ่มขึ้นมา แต่เมื่อโมเดลผ่านการเทรนด้วยข้อความจำนวนมหาศาล ตัวเลขในตารางจะถูกปรับจูนให้แม่นยำขึ้นเรื่อยๆ จนสะท้อนความหมายของคำได้อย่างลึกซึ้ง

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="850px">

แผนภาพลำดับขั้นตอนการทำงานทั้งหมดของโมเดล LLM (ตั้งแต่รับข้อความเข้าไป จนได้คำตอบออกมา) โดยเน้นจุดที่กำลังศึกษาอยู่ในหัวข้อนี้คือการสร้าง Token embeddings ครับ (ให้อ่านไล่จาก ด้านล่างขึ้นด้านบน ตามทิศทางลูกศร)

### <span style="color: #E2E8F0;">ลำดับการทำงานจากล่างขึ้นบน</span>

<br><span style="color: #64B5F6;"><b>1. Input text (ข้อความนำเข้า):</b></span>
	<br>- เริ่มต้นจากประโยคข้อความดิบที่เราป้อนเข้าไป คือ "This is an example."
<br><span style="color: #64B5F6;"><b>2. Tokenized text (การแบ่งโทเค็น):</b></span>
	<br>- ข้อความถูกหั่นออกเป็นคำศัพท์และเครื่องหมายย่อยๆ 5 ส่วน ได้แก่ ["This", "is", "an", "example", "."]
<br><span style="color: #64B5F6;"><b>3. Token IDs (รหัสโทเค็น):</b></span>
	<br>- คำศัพท์แต่ละส่วนถูกแปลงให้กลายเป็นรหัสตัวเลขจำนวนเต็มตามพจนานุกรม ได้แก่ 40134, 2052, 133, 389, และ 12
<br><span style="color: #64B5F6;"><b>4. Token embeddings (การฝังตัวโทเค็น - ส่วนที่เน้นในหัวข้อนี้):</b></span>
	<br>- รหัสตัวเลขแต่ละตัวจะถูกแปลงให้กลายเป็น ชุดเวกเตอร์ตัวเลขทศนิยม (ในภาพแทนด้วยกลุ่มกล่องสี่เหลี่ยม 3 ช่องต่อ 1 คำ ซึ่งสื่อถึงเวกเตอร์ขนาด 3 มิติ)
	<br>- ด้านซ้ายมีข้อความระบุไว้ว่า "This section covers creating input token embeddings" (หัวข้อนี้ครอบคลุมเรื่องการสร้าง Token embeddings สำหรับข้อมูลนำเข้า)
<br><span style="color: #64B5F6;"><b>5. GPT-like decoder-only transformer (ตัวโมเดลหลัก):</b></span>
	<br>- เวกเตอร์ทั้งหมดจะถูกส่งเข้าไปประมวลผลในสถาปัตยกรรมหลักของโมเดล LLM (เช่น GPT) เพื่อคำนวณความสัมพันธ์และทำนายผลลัพธ์
<br><span style="color: #64B5F6;"><b>6. Postprocessing steps (ขั้นตอนหลังการประมวลผล):</b></span>
	<br>- นำผลลัพธ์ตัวเลขที่ได้จากโมเดลมาแปลงกลับให้เป็นข้อความที่มนุษย์อ่านเข้าใจ
<br><span style="color: #64B5F6;"><b>7. Output text (ข้อความผลลัพธ์):</b></span>
	<br>- ได้คำตอบที่เป็นข้อความสมบูรณ์แสดงออกมาให้ผู้ใช้งานเห็น

- Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):
- *<small>สมมติว่าเรามีตัวอย่างข้อมูลนำเข้า (Input examples) 4 ตัว ที่มีรหัส Input IDs คือ 2, 3, 5 และ 1 (หลังจากผ่านการแบ่งโทเค็นเรียบร้อยแล้ว):*

In [42]:
input_ids = torch.tensor([2, 3, 5, 1])

- For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:
- <small>*เพื่อความเรียบง่ายและเข้าใจได้ง่าย สมมติว่าเรามีพจนานุกรมคำศัพท์ (Vocabulary) ขนาดเล็กที่มีคำศัพท์เพียง 6 คำ และเราต้องการสร้างเวกเตอร์ฝังตัว (Embeddings) ที่มีขนาดเท่ากับ 3 (มิติ):*

In [1]:
'''
โค้ดชุดนี้ทำหน้าที่ สร้างตารางค้นหาเวกเตอร์คำศัพท์ (Embedding Layer) 
ขนาดเล็กที่มี 6 คำศัพท์ โดยแต่ละคำจะถูกแทนที่ด้วยชุดตัวเลข 3 ค่า


# กำหนดขนาดของพจนานุกรมคำศัพท์ทั้งหมด (สมมติว่ามีคำศัพท์เพียง 6 คำ คือ ID 0 ถึง 5)
vocab_size = 6

# กำหนดขนาดมิติของเวกเตอร์สำหรับแทนความหมายของคำ 1 คำ (1 คำ = ตัวเลขทศนิยม 3 ตัว)
output_dim = 3

# ล็อกค่าเริ่มต้นของการสุ่มตัวเลข (Random Seed) เพื่อให้ได้ค่าตัวเลขสุ่มชุดเดิมทุกครั้งที่รันโค้ด (ตรงกับในหนังสือ)
torch.manual_seed(123)

# สร้างชั้น (Layer) Embedding ซึ่งเปรียบเสมือนตารางเมทริกซ์ขนาด 6 แถว x 3 คอลัมน์
# โดยตัวเลขข้างในจะถูกสุ่มขึ้นมาเป็นค่าเริ่มต้น และจะถูกปรับจูนให้แม่นยำขึ้นระหว่างการเทรนโมเดล
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

'''

'\nโค้ดชุดนี้ทำหน้าที่ สร้างตารางค้นหาเวกเตอร์คำศัพท์ (Embedding Layer) \nขนาดเล็กที่มี 6 คำศัพท์ โดยแต่ละคำจะถูกแทนที่ด้วยชุดตัวเลข 3 ค่า\n\n\n# กำหนดขนาดของพจนานุกรมคำศัพท์ทั้งหมด (สมมติว่ามีคำศัพท์เพียง 6 คำ คือ ID 0 ถึง 5)\nvocab_size = 6\n\n# กำหนดขนาดมิติของเวกเตอร์สำหรับแทนความหมายของคำ 1 คำ (1 คำ = ตัวเลขทศนิยม 3 ตัว)\noutput_dim = 3\n\n# ล็อกค่าเริ่มต้นของการสุ่มตัวเลข (Random Seed) เพื่อให้ได้ค่าตัวเลขสุ่มชุดเดิมทุกครั้งที่รันโค้ด (ตรงกับในหนังสือ)\ntorch.manual_seed(123)\n\n# สร้างชั้น (Layer) Embedding ซึ่งเปรียบเสมือนตารางเมทริกซ์ขนาด 6 แถว x 3 คอลัมน์\n# โดยตัวเลขข้างในจะถูกสุ่มขึ้นมาเป็นค่าเริ่มต้น และจะถูกปรับจูนให้แม่นยำขึ้นระหว่างการเทรนโมเดล\nembedding_layer = torch.nn.Embedding(vocab_size, output_dim)\n\n'

In [2]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

NameError: name 'torch' is not defined

- This would result in a 6x3 weight matrix:

In [44]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected layer, which is described in the supplementary code in [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)
- Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation

- To convert a token with id 3 into a 3-dimensional vector, we do the following:

In [45]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- Note that the above is the 4th row in the `embedding_layer` weight matrix
- To embed all four `input_ids` values above, we do

In [46]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- An embedding layer is essentially a look-up operation:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="850px">

- **You may be interested in the bonus content comparing embedding layers with regular linear layers: [../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

&nbsp;
## 2.8 Encoding word positions

- Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="850px">

- Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="850px">

- The BytePair encoder has a vocabulary size of 50,257:
- Suppose we want to encode the input tokens into a 256-dimensional vector representation:

In [47]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- If we sample data from the dataloader, we embed the tokens in each batch into a 256-dimensional vector
- If we have a batch size of 8 with 4 tokens each, this results in a 8 x 4 x 256 tensor:

In [48]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [49]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [50]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


- GPT-2 uses absolute position embeddings, so we just create another embedding layer:

In [51]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [52]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


- To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:

In [53]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


- In the initial phase of the input processing workflow, the input text is segmented into separate tokens
- Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="850px">

&nbsp;
## Summary and takeaways

See the [./dataloader.ipynb](./dataloader.ipynb) code notebook, which is a concise version of the data loader that we implemented in this chapter and will need for training the GPT model in upcoming chapters.

See [./exercise-solutions.ipynb](./exercise-solutions.ipynb) for the exercise solutions.

See the [Byte Pair Encoding (BPE) Tokenizer From Scratch](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) notebook if you are interested in learning how the GPT-2 tokenizer can be implemented and trained from scratch.